# Notebook Luyện Tập Thực Hành Machine Learning (Tuần 1 – Tuần 7)

Notebook này giúp bạn **tự tay code lại toàn bộ kiến thức cốt lõi** từ Tuần 1 đến Tuần 7 trên chính các bộ dữ liệu trong thư mục `2_Homework`.

### Cách sử dụng hiệu quả
1. **Chạy Cell Setup ngay bên dưới trước** (bắt buộc để nạp thư viện và hàm tìm đường dẫn `data_path`).
2. **Các Module 1 → 8 độc lập nhau**: Bạn có thể làm tuần tự từ đầu hoặc nhảy thẳng tới bất kỳ tuần nào muốn ôn tập.
3. Trong mỗi bài tập, tìm dòng `# TODO:` và tự viết code thay thế phần `raise NotImplementedError(...)` hoặc giá trị `None`.
4. Chạy **Cell Kiểm tra (`assert`)** ngay phía dưới mỗi bài để xác nhận logic, kích thước ma trận (`shape`), và độ chính xác.
5. Nếu cần tra cứu công thức hoặc cú pháp mẫu, xem lại tài liệu `Tong_Hop_Kien_Thuc_ML.md` ở cùng thư mục hoặc mở phần `<details>` gợi ý trong từng bài.

| Module | Nội dung ôn tập | Tuần tương ứng | Bộ dữ liệu sử dụng |
| :---: | :--- | :---: | :--- |
| **1** | Quy ước `(N, d)` vs `(d, N)`, làm sạch & chống Data Leakage | Xuyên suốt | `framingham.csv` |
| **2** | K-NN Regression & Linear Regression (`pinv`, QR, `LinearRegression`) | Tuần 1 | `real_estate.csv`, `fuel.txt` |
| **3** | Naïve Bayes (`GaussianNB` tự viết, BoW, `coo_matrix`, `CategoricalNB`) | Tuần 2 & 2B | `iris.csv`, `multinomial_nb_orders.csv` |
| **4** | Binary Logistic Regression (SGD tự viết, `sklearn`, `meshgrid`) | Tuần 3 | `Admission_Predict.csv` |
| **5** | Softmax Regression (`softmax_stable`, Mini-batch GD, MNIST `.gz`) | Tuần 4 | `iris.csv`, MNIST `.gz` |
| **6** | PCA tự cài đặt & Pipeline giữ 80% phương sai tránh rò rỉ | Tuần 5 | `iris.csv`, `pd_speech_features.csv` |
| **7** | LDA tự cài đặt ($S_W, S_B$) & So sánh PCA vs LDA | Tuần 6 | `iris.csv`, `pd_speech_features.csv` |
| **8** | Perceptron (PLA `{+1, -1}`) & Capstone So sánh Mô hình | Tuần 7 | Dữ liệu tổng hợp, `Analysis-Portfolio-Task-Data.csv`, `sonar.all-data.csv` |


In [9]:
from pathlib import Path
import gzip
import struct
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.sparse import coo_matrix

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression, LogisticRegression, Perceptron
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, CategoricalNB
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

def find_root_dir():
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "DVA_24_week1_knn").exists():
            return p
    raise FileNotFoundError("Không tìm thấy thư mục DVA_24_week1_knn từ vị trí hiện tại!")

ROOT_DIR = find_root_dir()

def data_path(*parts):
    path = ROOT_DIR.joinpath(*parts)
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")
    return path

print("ROOT_DIR =", ROOT_DIR)
print("Setup hoàn tất!")

ROOT_DIR = D:\Data\1_University\1_Specialized_Subjects\9_MachineLearning\2_Homework
Setup hoàn tất!


---
## Module 1: Quy Ước Ma Trận `(N, d)` vs `(d, N)` & Tiền Xử Lý Chống Data Leakage

### Bài 1.1: Chuẩn hóa hình dạng ma trận gắn Bias (`add_bias_row` vs `add_bias_col`)
- Trong **Week 1 Linear Regression** (mẫu theo hàng `(N, d)`), ta ghép **cột số 1** vào bên trái $\implies (N, d + 1)$.
- Trong **Week 3, 4, 7 NumPy SGD/PLA** (mẫu theo cột `(d, N)`), ta chuyển vị $X^T$ và ghép **hàng số 1** lên trên $\implies (d + 1, N)$.

Hãy cài đặt 2 hàm chuyển đổi ma trận bên dưới.

<details><summary>💡 Gợi ý nhanh Bài 1.1</summary>

- `to_row_samples_with_bias(X)`: dùng `np.hstack([np.ones((X.shape[0], 1)), X])`.
- `to_col_samples_with_bias(X)`: dùng `np.vstack([np.ones((1, X.shape[0])), X.T])`.

</details>


In [12]:
def to_row_samples_with_bias(X):
    """Nhận X shape (N, d) -> trả về X_bar shape (N, d + 1) có cột đầu toàn 1."""
    # TODO: Viết code tại đây
    bias = np.ones((X.shape[0], 1))
    Xbar = np.concatenate((bias, X), axis=1)
    return Xbar

def to_col_samples_with_bias(X):
    """Nhận X shape (N, d) -> trả về X_bar shape (d + 1, N) có hàng đầu toàn 1."""
    # TODO: Viết code tại đây
    bias = np.ones((1, X.shape[0]))
    Xbar = np.concatenate((bias, X.T), axis=0)
    return Xbar

In [13]:
X_demo = np.array([[2.0, 3.0, 4.0], [5.0, 6.0, 7.0]])
X_row = to_row_samples_with_bias(X_demo)
X_col = to_col_samples_with_bias(X_demo)
assert X_row.shape == (2, 4) and np.all(X_row[:, 0] == 1)
assert X_col.shape == (4, 2) and np.all(X_col[0, :] == 1)
assert np.allclose(X_row.T, X_col)
print("✅ Đạt Bài 1.1! Shapes:", X_row.shape, "và", X_col.shape)

✅ Đạt Bài 1.1! Shapes: (2, 4) và (4, 2)


### Bài 1.2: Tiền xử lý sạch không rò rỉ trên `framingham.csv`
Thực hiện đúng quy trình chuẩn cho `DVA_24_week3_Th3LogisticReg/framingham.csv` (nhãn mục tiêu là cột `"TenYearCHD"`):
1. Đọc CSV và loại bỏ các hàng chứa giá trị thiếu (`dropna()`).
2. Tách `X` (15 đặc trưng) và `y` (`"TenYearCHD"`).
3. Chia train/test tỷ lệ `70:30` (`test_size=0.3`), bật `stratify=y` và `random_state=42`.
4. Dùng `StandardScaler` chỉ `.fit_transform()` trên `X_train` và `.transform()` trên `X_test`.


In [ ]:
def prepare_framingham_data():
    df = pd.read_csv(data_path("DVA_24_week3_Th3LogisticReg", "framingham.csv"))
    # TODO: Bỏ missing, tách X/y, chia stratified 70:30 (random_state=42), scale đúng cách
    raise NotImplementedError("Hãy hoàn thiện prepare_framingham_data")
    return X_train_s, X_test_s, y_train, y_test, scaler

In [ ]:
X_tr_s, X_te_s, y_tr, y_te, sc = prepare_framingham_data()
assert X_tr_s.shape[1] == 15 and len(X_tr_s) + len(X_te_s) == 3656
assert np.allclose(X_tr_s.mean(axis=0), 0, atol=1e-10)
assert abs(y_tr.mean() - y_te.mean()) < 0.005, "Chú ý bật stratify=y khi chia!"
print(f"✅ Đạt Bài 1.2! Train shape={X_tr_s.shape}, Test shape={X_te_s.shape}")

---
## Module 2 (Tuần 1): K-NN Regression & Linear Regression

### Bài 2.1: Tự viết K-NN Regression trên `real_estate.csv`
Hoàn thiện hai hàm `calc_distances(X_train, x_query)` (hỗ trợ cả mảng 1D lẫn ma trận 2D `(N, d)` — nhớ `axis=1`!) và `knn_reg_predict(X_train, y_train, X_test, k)`.


In [ ]:
def calc_distances(X_train, x_query):
    # TODO: Nếu X_train.ndim == 1 trả về |X_train - x_query|, ngược lại dùng np.linalg.norm(..., axis=1)
    raise NotImplementedError("Hãy hoàn thiện calc_distances")

def knn_reg_predict(X_train, y_train, X_test, k=5):
    # TODO: Với từng mẫu trong X_test, tìm k chỉ số gần nhất rồi lấy trung bình y_train của k mẫu đó
    raise NotImplementedError("Hãy hoàn thiện knn_reg_predict")

In [ ]:
df_re = pd.read_csv(data_path("DVA_24_week1_knn", "real_estate.csv"))
X_re = df_re.iloc[:, 1:-1].to_numpy()
y_re = df_re.iloc[:, -1].to_numpy()
X_re_tr, X_re_te, y_re_tr, y_re_te = train_test_split(X_re, y_re, test_size=0.2, random_state=42)

y_re_pred = knn_reg_predict(X_re_tr, y_re_tr, X_re_te, k=18)
mse_re = mean_squared_error(y_re_te, y_re_pred)
r2_re = r2_score(y_re_te, y_re_pred)
assert calc_distances(np.array([1.0, 4.0, 10.0]), 3.0).tolist() == [2.0, 1.0, 7.0]
assert y_re_pred.shape == y_re_te.shape and r2_re > 0.65
print(f"✅ Đạt Bài 2.1! Real Estate K=18 -> MSE={mse_re:.4f}, R2={r2_re:.4f}")

### Bài 2.2: Linear Regression bằng Nghịch đảo giả (`pinv`) & QR trên `fuel.txt`
Theo bài tập B.1 Tuần 1, ta tạo 4 đặc trưng `[Tax, Income, Miles, Drivers/Pop]` và biến mục tiêu `1000 * FuelC / Pop`, lấy 40 mẫu đầu làm train và phần còn lại làm test.
Hoàn thiện:
1. `fit_linear_pinv(X_train, y_train)`: ghép cột `1` ở bên trái rồi tính $w = (\bar{X}^T\bar{X})^\dagger \bar{X}^T y$.
2. `fit_linear_qr(X_train, y_train)`: ghép cột `1` rồi phân tích $\bar{X} = QR$ bằng `np.linalg.qr` và giải $Rw = Q^T y$ bằng `np.linalg.solve`.


In [ ]:
def fit_linear_pinv(X_train, y_train):
    # TODO: Ghép cột 1 và tính w bằng np.linalg.pinv
    raise NotImplementedError("Hãy hoàn thiện fit_linear_pinv")

def fit_linear_qr(X_train, y_train):
    # TODO: Ghép cột 1 và tính w bằng np.linalg.qr + np.linalg.solve
    raise NotImplementedError("Hãy hoàn thiện fit_linear_qr")

In [ ]:
df_f = pd.read_csv(data_path("DVA_24_week1_knn", "fuel.txt"))
X_f = np.column_stack([
    df_f["Tax"].to_numpy(),
    df_f["Income"].to_numpy(),
    df_f["Miles"].to_numpy(),
    df_f["Drivers"].to_numpy() / df_f["Pop"].to_numpy(),
])
y_f = 1000.0 * df_f["FuelC"].to_numpy() / df_f["Pop"].to_numpy()
X_f_tr, X_f_te, y_f_tr, y_f_te = X_f[:40], X_f[40:], y_f[:40], y_f[40:]

w_pinv = fit_linear_pinv(X_f_tr, y_f_tr)
w_qr = fit_linear_qr(X_f_tr, y_f_tr)
reg_sk = LinearRegression().fit(X_f_tr, y_f_tr)

assert np.allclose(w_pinv, w_qr)
assert np.allclose(w_pinv[0], reg_sk.intercept_) and np.allclose(w_pinv[1:], reg_sk.coef_)
y_f_pred = np.hstack([np.ones((X_f_te.shape[0], 1)), X_f_te]) @ w_qr
print(f"✅ Đạt Bài 2.2! Khớp tuyệt đối sklearn. MSE={mean_squared_error(y_f_te, y_f_pred):.2f}, MAE={mean_absolute_error(y_f_te, y_f_pred):.2f}")

---
## Module 3 (Tuần 2 & 2B): Naïve Bayes (Gaussian, Sparse `coo_matrix`, Categorical)

### Bài 3.1: Tự cài đặt `GaussianNB` bám sát code mẫu (`mu_list`, `std_list`, `pi_list`, `predict_NB_gaussian_class`)
Hoàn thiện 2 hàm bám sát cấu trúc code mẫu Tuần 2 (`DinhVietAnh_24001964_week2_nb.ipynb`):
- `fit_NB_gaussian_params(train_df, target_col="variety")`: dùng `train_df.groupby(target_col, sort=True)` + `np.split(..., [1, 2], axis=0)` để tạo `mu_list`, `std_list` (dùng `ddof=0` để khớp `sklearn`), và `pi_list` bằng `value_counts().sort_index().values / len(train_df)`.
- `predict_NB_gaussian_class(X, mu_list, std_list, pi_list)`: nhận 1 mẫu `X` dạng `(d, 1)` (hoặc `(d,)`), nhân dồn `stats.norm.pdf(x=X[j], loc=mu_list[p][0][j], scale=std_list[p][0][j])` với `pi_list[p]` cho từng lớp `p`, trả về `np.argmax(scores_list)`.

<details><summary>💡 Gợi ý nhanh Bài 3.1</summary>

- `mu_list[p]` có shape `(1, d)` nên lấy trung bình chiều `j` của lớp `p` bằng `mu_list[p][0][j]`.
- Nhớ `sort=True` và `.sort_index()` để thứ tự 3 lớp luôn là `0: Setosa`, `1: Versicolor`, `2: Virginica`.

</details>


In [7]:
def fit_NB_gaussian_params(train_df, target_col="variety"):
    # TODO: Tính mu_list, std_list (ddof=0), pi_list theo đúng cấu trúc groupby + np.split
    mu_list = np.split(train_df.groupby(target_col, sort=True).mean().values, [1, 2], axis = 0)
    std_list = np.split(train_df.groupby(target_col, sort=True).std(ddof=0).values, [1, 2], axis=0)
    pi_list = train_df.groupby(target_col, sort=True).count_value().values / len(train_df)
    return mu_list, std_list, pi_list

def predict_NB_gaussian_class(X, mu_list, std_list, pi_list):
    # TODO: Tính scores_list cho từng lớp p bằng stats.norm.pdf và trả về np.argmax(scores_list)
    scores = []
    classes = len(mu_list)
    for c in classes:
        score = pi_list[c]
        for j in range(len(X)):
            score *= stats.norm.pdf(x=X[j], loc = mu_list[c][0][j], scale = std_list[c][0][j])
        scores.append(score)
    return np.argmax(scores)

In [8]:
df_ir = pd.read_csv(data_path("DVA_24_week2_NaiveBayes", "iris.csv"))
train_df_ir, test_df_ir = train_test_split(df_ir, test_size=0.3, stratify=df_ir["variety"], random_state=42)
mu_list, std_list, pi_list = fit_NB_gaussian_params(train_df_ir, target_col="variety")

X_ir_te = test_df_ir.iloc[:, :4].to_numpy()
y_ir_te = test_df_ir["variety"].map({"Setosa": 0, "Versicolor": 1, "Virginica": 2}).to_numpy()
y_ir_tr = train_df_ir["variety"].map({"Setosa": 0, "Versicolor": 1, "Virginica": 2}).to_numpy()

pred_man = np.array([predict_NB_gaussian_class(x.reshape(-1, 1), mu_list, std_list, pi_list) for x in X_ir_te])
pred_sk = GaussianNB().fit(train_df_ir.iloc[:, :4].to_numpy(), y_ir_tr).predict(X_ir_te)
assert len(mu_list) == 3 and mu_list[0].shape == (1, 4)
assert np.array_equal(pred_man, pred_sk), "Dự đoán tự viết phải khớp hoàn toàn sklearn GaussianNB!"
print(f"✅ Đạt Bài 3.1! Accuracy = {accuracy_score(y_ir_te, pred_man):.4f} (khớp 100% sklearn)")

NameError: name 'pd' is not defined

### Bài 3.2: Ma trận thưa 1-based `coo_matrix` & `CategoricalNB` với `OrdinalEncoder`
1. Từ dữ liệu thưa đánh số **1-based** `(doc_id, word_id, count)`, tạo ma trận `coo_matrix` đúng chuẩn **0-based** kích thước `(3, 4)` (nhớ `- 1`!).
2. Với tập `DVA_24_week2_NaiveBayes/multinomial_nb_orders.csv` (`y = "cuisine"`): chia train/test 80:20 (`stratify=y`), mã hóa đặc trưng chữ bằng `OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)` rồi huấn luyện `CategoricalNB(alpha=1.0)`.


In [ ]:
def build_sparse_from_1based(doc_ids, word_ids, counts, n_docs, n_words):
    # TODO: Tạo scipy.sparse.coo_matrix có shape=(n_docs, n_words) nhớ chuyển 1-based -> 0-based
    raise NotImplementedError("Hãy hoàn thiện build_sparse_from_1based")

def train_orders_categorical_nb():
    df_ord = pd.read_csv(data_path("DVA_24_week2_NaiveBayes", "multinomial_nb_orders.csv"))
    # TODO: Tách X, y; chia 80:20 stratify=y, random_state=42; fit OrdinalEncoder trên train; fit CategoricalNB(alpha=1.0)
    raise NotImplementedError("Hãy hoàn thiện train_orders_categorical_nb")
    return acc, model, encoder

In [ ]:
M = build_sparse_from_1based([1, 2, 3], [1, 4, 2], [5, 2, 7], n_docs=3, n_words=4)
assert M.shape == (3, 4) and M.toarray()[0, 0] == 5 and M.toarray()[1, 3] == 2 and M.toarray()[2, 1] == 7
acc_cnb, _, _ = train_orders_categorical_nb()
assert acc_cnb > 0.45
print(f"✅ Đạt Bài 3.2! Sparse OK | Orders CategoricalNB Test Acc = {acc_cnb:.4f}")

---
## Module 4 (Tuần 3): Binary Logistic Regression

### Bài 4.1: Tự viết `sigmoid` và `logistic_sgd` theo quy ước cột `(d + 1, N)`
Áp dụng trên `DVA_24_week3_Th3LogisticReg/Admission_Predict.csv` (nhị phân hóa `Chance of Admit >= 0.75`).
- Lưu ý: $X_{\text{bar}}$ đầu vào của `logistic_sgd` có shape `(d + 1, N)`, mỗi mẫu là một cột `X_bar[:, [i]]` `(d + 1, 1)`.
- Công thức cập nhật mỗi mẫu: $w_{\text{new}} = w + \eta (y_i - \sigma(w^T x_i)) x_i$.


In [ ]:
def sigmoid(s):
    # TODO: Clip s trong [-500, 500] rồi trả về 1 / (1 + exp(-s))
    raise NotImplementedError("Hãy hoàn thiện sigmoid")

def logistic_sgd(X_bar, y, w_init, eta=0.05, tol=1e-4, max_count=10000, seed=42):
    # TODO: Cài đặt SGD duyệt hoán vị từng mẫu, dừng sớm nếu ||w_new - w_hist[-20]|| < tol hoặc đủ max_count
    raise NotImplementedError("Hãy hoàn thiện logistic_sgd")

In [ ]:
df_adm = pd.read_csv(data_path("DVA_24_week3_Th3LogisticReg", "Admission_Predict.csv"))
X_a = df_adm.iloc[:, 1:8].to_numpy()
y_a = (df_adm.iloc[:, 8].to_numpy() >= 0.75).astype(int)
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(X_a, y_a, test_size=0.2, stratify=y_a, random_state=42)
sc_a = StandardScaler()
Xa_tr_s, Xa_te_s = sc_a.fit_transform(Xa_tr), sc_a.transform(Xa_te)

Xb_tr = np.vstack([np.ones((1, len(Xa_tr_s))), Xa_tr_s.T])
Xb_te = np.vstack([np.ones((1, len(Xa_te_s))), Xa_te_s.T])
w_a = logistic_sgd(Xb_tr, ya_tr, np.zeros((Xb_tr.shape[0], 1)), eta=0.05, max_count=10000)
pred_a = (sigmoid((w_a.T @ Xb_te).ravel()) >= 0.5).astype(int)
acc_a = accuracy_score(ya_te, pred_a)
assert np.allclose(sigmoid(np.array([0.0, 1000.0, -1000.0])), [0.5, 1.0, 0.0], atol=1e-6)
assert acc_a >= 0.80
print(f"✅ Đạt Bài 4.1! Admission Logistic SGD Acc = {acc_a:.4f}")

### Bài 4.2: Sklearn `LogisticRegression` (không cảnh báo deprecation) & Tạo lưới `meshgrid` 2D
- Trên `sklearn >= 1.8`, dùng **`C=np.inf`** thay cho `penalty=None` khi muốn bỏ điều chuẩn.
- Hoàn thiện hàm `make_boundary_grid(model, X_2d, step=100)` dùng `np.meshgrid` và `model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)` để chuẩn bị dữ liệu vẽ `plt.contourf`.


In [ ]:
def fit_unregularized_lr(X_train, y_train):
    # TODO: Trả về LogisticRegression đã fit không dùng regularization (dùng C=np.inf, max_iter=1000)
    raise NotImplementedError("Hãy hoàn thiện fit_unregularized_lr")

def make_boundary_grid(model, X_2d, n_points=100):
    # TODO: Tạo xx, yy bằng np.meshgrid rồi tính ma trận dự đoán Z cùng shape với xx
    raise NotImplementedError("Hãy hoàn thiện make_boundary_grid")
    return xx, yy, Z

In [ ]:
df_adm2 = pd.read_csv(data_path("DVA_24_week3_Th3LogisticReg", "Admission_Predict.csv"))
X_2d = StandardScaler().fit_transform(df_adm2[["GRE Score", "CGPA"]].to_numpy())
y_2d = (df_adm2.iloc[:, 8].to_numpy() >= 0.75).astype(int)
clf_2d = fit_unregularized_lr(X_2d, y_2d)
xx, yy, Z = make_boundary_grid(clf_2d, X_2d, n_points=80)
assert xx.shape == (80, 80) and Z.shape == (80, 80)
print(f"✅ Đạt Bài 4.2! 2D Train Acc = {clf_2d.score(X_2d, y_2d):.4f}, Grid shape = {Z.shape}")

---
## Module 5 (Tuần 4): Softmax Regression (Đa lớp) & Dữ liệu MNIST `.gz`

### Bài 5.1: Cài đặt `one_hot_columns`, `softmax_stable`, `cross_entropy_loss` & `softmax_minibatch_gd`
- `Z` có shape `(C, N)` (mỗi cột là điểm logit của $C$ lớp cho 1 mẫu).
- Nhớ trừ `np.max(Z, axis=0, keepdims=True)` trong `softmax_stable` và `np.clip(A, 1e-15, 1.0)` trong `cross_entropy_loss`.
- Gradient theo mini-batch: $\nabla_W J = \frac{1}{B} \bar{X}_b (A_b - Y_b)^T$.


In [ ]:
def one_hot_columns(y, C):
    # TODO: Trả về ma trận one-hot shape (C, len(y))
    raise NotImplementedError("Hãy hoàn thiện one_hot_columns")

def softmax_stable(Z):
    # TODO: Softmax ổn định số học theo axis=0
    raise NotImplementedError("Hãy hoàn thiện softmax_stable")

def cross_entropy_loss(Y, A):
    # TODO: Tính Categorical Cross-Entropy trung bình trên N mẫu (nhớ np.clip)
    raise NotImplementedError("Hãy hoàn thiện cross_entropy_loss")

def softmax_minibatch_gd(X_bar, y, C, eta=0.1, epochs=150, batch_size=32, seed=42):
    # TODO: Huấn luyện W shape (d+1, C) bằng Mini-batch Gradient Descent
    raise NotImplementedError("Hãy hoàn thiện softmax_minibatch_gd")

In [ ]:
Z_test = np.array([[1000.0, 1.0], [1000.0, 2.0], [999.0, 3.0]])
P_test = softmax_stable(Z_test)
assert not np.isnan(P_test).any() and np.allclose(P_test.sum(axis=0), 1.0)
assert cross_entropy_loss(one_hot_columns(np.array([0, 2]), 3), P_test) > 0

df_ir5 = pd.read_csv(data_path("DVA_24_week4_Th4", "iris.csv"))
X5 = df_ir5.iloc[:, :4].to_numpy()
y5 = df_ir5["variety"].map({"Setosa": 0, "Versicolor": 1, "Virginica": 2}).to_numpy()
X5_tr, X5_te, y5_tr, y5_te = train_test_split(X5, y5, test_size=0.3, stratify=y5, random_state=42)
Xb5_tr = np.vstack([np.ones((1, len(X5_tr))), X5_tr.T])
Xb5_te = np.vstack([np.ones((1, len(X5_te))), X5_te.T])

W5 = softmax_minibatch_gd(Xb5_tr, y5_tr, C=3, eta=0.1, epochs=300, batch_size=16)
acc_sm = accuracy_score(y5_te, np.argmax(W5.T @ Xb5_te, axis=0))
assert acc_sm >= 0.88
print(f"✅ Đạt Bài 5.1! Softmax stable OK | Iris Softmax Test Acc = {acc_sm:.4f}")

### Bài 5.2: Nạp file nhị phân `.gz` chuẩn IDX của MNIST
Đọc cặp file `train-images-idx3-ubyte.gz` (header 16 byte: `>IIII`) và `train-labels-idx1-ubyte.gz` (header 8 byte: `>II`) từ `DVA_24_week4_Th4`, chuẩn hóa pixel về `[0, 1]` bằng cách chia `255.0`.


In [ ]:
def load_mnist_gz(img_path, lbl_path):
    # TODO: Mở bằng gzip.open, unpack header 16 byte / 8 byte bằng struct.unpack, đọc np.uint8 & chia 255.0
    raise NotImplementedError("Hãy hoàn thiện load_mnist_gz")
    return X, y

In [ ]:
X_m_tr, y_m_tr = load_mnist_gz(
    data_path("DVA_24_week4_Th4", "train-images-idx3-ubyte.gz"),
    data_path("DVA_24_week4_Th4", "train-labels-idx1-ubyte.gz"),
)
X_m_te, y_m_te = load_mnist_gz(
    data_path("DVA_24_week4_Th4", "t10k-images-idx3-ubyte.gz"),
    data_path("DVA_24_week4_Th4", "t10k-labels-idx1-ubyte.gz"),
)
assert X_m_tr.shape == (60000, 784) and X_m_te.shape == (10000, 784)
assert X_m_tr.max() <= 1.0 and X_m_tr.min() >= 0.0
clf_m = LogisticRegression(max_iter=300).fit(X_m_tr[:2000], y_m_tr[:2000])
acc_m = clf_m.score(X_m_te[:500], y_m_te[:500])
assert acc_m >= 0.80
print(f"✅ Đạt Bài 5.2! MNIST loaded {X_m_tr.shape}, Subset LR Test Acc = {acc_m:.4f}")

---
## Module 6 (Tuần 5): Principal Component Analysis (PCA) & Chống Data Leakage

### Bài 6.1: Tự cài đặt PCA 5 bước khớp hoàn toàn với `sklearn.decomposition.PCA`
Với ma trận đã chuẩn hóa `Z` (shape `(N, d)`, `ddof=0`):
1. Tính ma trận hiệp phương sai $\Sigma = \frac{1}{N} Z^T Z$.
2. Dùng `np.linalg.eigh(Sigma)` để lấy trị riêng & vector riêng, sau đó **sắp xếp giảm dần** (`[::-1]`).
3. Trả về `(Z_projected, explained_ratio)` cho `n_components` chiều đầu tiên.


In [ ]:
def manual_pca(Z, n_components=2):
    # TODO: Tính ma trận hiệp phương sai (1/N) * Z.T @ Z, eigh, sắp xếp giảm dần, chiếu và tính tỷ lệ phương sai
    raise NotImplementedError("Hãy hoàn thiện manual_pca")
    return Z_proj, explained_ratio

In [ ]:
df_ir6 = pd.read_csv(data_path("DVA_24_week5_Th5", "iris.csv"))
Z_ir6 = StandardScaler().fit_transform(df_ir6.iloc[:, :4].to_numpy())
Z_man, ratio_man = manual_pca(Z_ir6, n_components=2)
pca_sk = PCA(n_components=2).fit(Z_ir6)
assert Z_man.shape == (150, 2)
assert np.allclose(ratio_man, pca_sk.explained_variance_ratio_)
print("✅ Đạt Bài 6.1! Explained variance ratio khớp sklearn:", ratio_man.round(4))

### Bài 6.2: Pipeline PCA an toàn giữ 80% phương sai trên `pd_speech_features.csv`
- Đọc `DVA_24_week5_Th5/pd_speech_features.csv`, bỏ cột `"id"` và tách nhãn `"class"`.
- Chia train/test 70:30 (`stratify=y`, `random_state=42`) **TRƯỚC** khi gọi `StandardScaler` và `PCA(n_components=0.80, svd_solver="full")`.
- Huấn luyện `LogisticRegression(max_iter=1000)` trên dữ liệu đã giảm chiều.


In [ ]:
def run_parkinson_pca_pipeline():
    df_pd = pd.read_csv(data_path("DVA_24_week5_Th5", "pd_speech_features.csv"))
    # TODO: Thực hiện split -> StandardScaler(fit train) -> PCA(0.80, full, fit train) -> LogisticRegression
    raise NotImplementedError("Hãy hoàn thiện run_parkinson_pca_pipeline")
    return pca.n_components_, acc_test

In [ ]:
n_comp, acc_pd = run_parkinson_pca_pipeline()
assert 40 <= n_comp <= 70 and acc_pd >= 0.75
print(f"✅ Đạt Bài 6.2! Parkinson 753D -> {n_comp}D (giữ 80% var), LR Test Acc = {acc_pd:.4f}")

---
## Module 7 (Tuần 6): Linear Discriminant Analysis (LDA) & So Sánh PCA vs LDA

### Bài 7.1: Tự cài đặt LDA đa lớp từ đầu trên `iris.csv`
Tính:
- $S_W = \sum_c (X_c - m_c)^T (X_c - m_c)$
- $S_B = \sum_c n_c (m_c - m)(m_c - m)^T$ (với $(m_c - m)$ shape `(d, 1)`)
- Giải `np.linalg.eig(np.linalg.pinv(S_W) @ S_B)` $\implies$ **nhớ lấy `.real` và sắp xếp giảm dần theo trị riêng**, chọn `n_components` ($\le C - 1$) cột đầu lập ma trận chiếu $W$.


In [ ]:
def manual_lda_fit(X, y, n_components=2):
    # TODO: Tính S_W, S_B, giải trị riêng tổng quát (nhớ .real + sort giảm dần), trả về W shape (d, n_components)
    raise NotImplementedError("Hãy hoàn thiện manual_lda_fit")
    return W

In [ ]:
df_ir7 = pd.read_csv(data_path("DVA_24_week2_NaiveBayes", "iris.csv"))
X7 = df_ir7.iloc[:, :4].to_numpy()
y7 = df_ir7["variety"].map({"Setosa": 0, "Versicolor": 1, "Virginica": 2}).to_numpy()
X7_tr, X7_te, y7_tr, y7_te = train_test_split(X7, y7, test_size=0.3, stratify=y7, random_state=42)

W_lda = manual_lda_fit(X7_tr, y7_tr, n_components=2)
acc_lda_man = LogisticRegression(max_iter=300).fit(X7_tr @ W_lda, y7_tr).score(X7_te @ W_lda, y7_te)
lda_sk = LinearDiscriminantAnalysis(n_components=2).fit(X7_tr, y7_tr)
acc_lda_sk = LogisticRegression(max_iter=300).fit(lda_sk.transform(X7_tr), y7_tr).score(lda_sk.transform(X7_te), y7_te)
assert W_lda.shape == (4, 2) and acc_lda_man >= 0.90
print(f"✅ Đạt Bài 7.1! Iris LDA 2D+LR: Manual Acc={acc_lda_man:.4f} | Sklearn Acc={acc_lda_sk:.4f}")

### Bài 7.2: So sánh Original vs PCA (2D) vs LDA (1D) trên Parkinson
Trên `DVA_24_week6_Th6/pd_speech_features.csv` (bài toán nhị phân $C=2$ nên LDA tối đa **1 chiều**!):
So sánh độ chính xác trên tập test của `LogisticRegression(max_iter=1000)` trên 3 không gian:
1. Gốc đã `StandardScaler` (753 chiều).
2. `PCA(n_components=2)` sau khi scale.
3. `LinearDiscriminantAnalysis(n_components=1)` sau khi scale (hoặc làm trực tiếp bộ phân loại).


In [ ]:
def compare_parkinson_dim_reduction():
    df = pd.read_csv(data_path("DVA_24_week6_Th6", "pd_speech_features.csv"))
    # TODO: Chia 70:30 stratify=y random_state=42 -> scale -> tính dictionary kết quả {"Original": ..., "PCA_2D": ..., "LDA_1D": ...}
    raise NotImplementedError("Hãy hoàn thiện compare_parkinson_dim_reduction")
    return results

In [ ]:
res_dim = compare_parkinson_dim_reduction()
assert set(res_dim.keys()) == {"Original", "PCA_2D", "LDA_1D"}
print("✅ Đạt Bài 7.2! Kết quả Parkinson:", {k: round(v, 4) for k, v in res_dim.items()})

---
## Module 8 (Tuần 7): Perceptron Learning Algorithm (PLA) & Capstone Tổng Hợp

### Bài 8.1: Tự cài đặt PLA chuẩn `{+1, -1}` có chốt chặn `max_epochs`
- Nhãn bắt buộc thuộc **`{+1, -1}`**.
- Nếu `np.where((w.T @ X_bar).ravel() >= 0, 1, -1)` khớp hoàn toàn `y`, trả về `(w, True)`.
- Nếu gặp mẫu sai `i`: cập nhật `w = w + y[i] * X_bar[:, [i]]`.
- Nếu hết `max_epochs` chưa tách hết, dừng an toàn và trả về `(w, False)` để tránh treo máy!


In [ ]:
def pla_fit(X_bar, y, w_init, max_epochs=500, seed=42):
    # TODO: Cài đặt PLA trên X_bar shape (d+1, N), y thuộc {+1, -1}; trả về (w, converged_bool)
    raise NotImplementedError("Hãy hoàn thiện pla_fit")
    return w, converged

In [ ]:
rng8 = np.random.default_rng(42)
# 1. Tập khả tách tuyến tính -> phải hội tụ 100%
X_sep = np.vstack([rng8.normal(-3, 0.5, (20, 2)), rng8.normal(3, 0.5, (20, 2))])
y_sep = np.array([-1] * 20 + [1] * 20)
Xb_sep = np.vstack([np.ones((1, len(X_sep))), X_sep.T])
w_sep, conv_sep = pla_fit(Xb_sep, y_sep, np.zeros((3, 1)), max_epochs=200)
assert conv_sep is True

# 2. Tập chồng lấn không khả tách -> phải dừng an toàn sau max_epochs
X_ov = np.vstack([rng8.normal(0, 1.0, (20, 2)), rng8.normal(0.1, 1.0, (20, 2))])
Xb_ov = np.vstack([np.ones((1, len(X_ov))), X_ov.T])
w_ov, conv_ov = pla_fit(Xb_ov, y_sep, np.zeros((3, 1)), max_epochs=50)
assert conv_ov is False
print("✅ Đạt Bài 8.1! Khả tách hội tụ =", conv_sep, "| Không khả tách dừng an toàn =", (not conv_ov))

### Bài 8.2 (Capstone): Fresco & Sonar — Tiền xử lý, `Perceptron` vs `LogisticRegression` vs `GaussianNB`, Diễn giải `coef_`
Thực hiện trên `DVA_24_week7_Th7/Analysis-Portfolio-Task-Data.csv` (phân loại `Shopping Basket > 50`) và `DVA_24_week7_Th7/sonar.all-data.csv`:
1. Với **Fresco**: bỏ `CustomerID`, map `"Male": 1, "Female": 0`, dùng `pd.get_dummies(..., columns=["Store Type"], drop_first=True, dtype=int)`, tách `X/y` (`Shopping Basket > 50`), chia `train_size=60` (`stratify=y`, `random_state=42`), chuẩn hóa bằng `StandardScaler`, huấn luyện `LogisticRegression(max_iter=1000)` và tìm **tên đặc trưng có `|coef_|` lớn nhất**.
2. Với **Sonar**: đọc không header (`header=None`), đổi cột nhãn cuối `'M' -> 1, 'R' -> 0` (hoặc `{+1, -1}`), chia `80:20` (`stratify=y`, `random_state=42`), chuẩn hóa và trả về bảng điểm `Accuracy` của 3 mô hình `Perceptron`, `LogisticRegression`, `GaussianNB`.


In [ ]:
def fresco_top_feature():
    df = pd.read_csv(data_path("DVA_24_week7_Th7", "Analysis-Portfolio-Task-Data.csv"))
    # TODO: Làm sạch, map Gender, get_dummies Store Type, chia 60/rest stratify=y, scale, fit LR, trả về (top_feat_name, test_acc)
    raise NotImplementedError("Hãy hoàn thiện fresco_top_feature")
    return top_feat_name, test_acc

def sonar_benchmark():
    df = pd.read_csv(data_path("DVA_24_week7_Th7", "sonar.all-data.csv"), header=None)
    # TODO: Chuyển cột 60 {'M': 1, 'R': 0}, chia 80:20 stratify=y random_state=42, scale, đánh giá 3 mô hình
    raise NotImplementedError("Hãy hoàn thiện sonar_benchmark")
    return scores

In [ ]:
top_feat, acc_fr = fresco_top_feature()
sonar_scores = sonar_benchmark()
assert isinstance(top_feat, str) and acc_fr >= 0.70
assert set(sonar_scores.keys()) == {"Perceptron", "LogisticRegression", "GaussianNB"}
print(f"✅ Đạt Bài 8.2! Fresco top feature = '{top_feat}' (Test Acc={acc_fr:.4f})")
print("   Sonar Benchmark Acc:", {k: round(v, 4) for k, v in sonar_scores.items()})